# Exercises

In this section we have two exercises:
1. Implement the polynomial kernel.
2. Implement the multiclass C-SVM.

## Polynomial kernel

You need to extend the ``build_kernel`` function and implement the polynomial kernel if the ``kernel_type`` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}

In [31]:
def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    elif kernel_type == 'poly':
        d = 3 # Setting the polynomial degree
        kernel = kernel ** d
    return kernel

## Implement a multiclass C-SVM

Use the classification method that we used in notebook 7.3 and IRIS dataset to build a multiclass C-SVM classifier. Most implementation is about a function that will return the proper data set that need to be used for the prediction. You need to implement:
- ``choose_set_for_label``
- ``get_labels_count``

In [32]:
def choose_set_for_label(data_set, label):
    train_data_set, test_data_set, original_train_labels, original_test_labels = data_set
    
    # Create binary labels: +1 for the target class, -1 for all others
    train_labels = np.where(original_train_labels == label, 1, -1).astype(float)
    test_labels = np.where(original_test_labels == label, 1, -1).astype(float)
    
    return train_data_set, test_data_set, train_labels, test_labels

In [33]:
def get_labels_count(data_set):
    _, _, original_train_labels, _ = data_set
    labels_count = len(np.unique(original_train_labels))
    return labels_count

Use the code that we have implemented earlier:

In [34]:
import cvxopt

def train(train_data_set, train_labels, kernel_type='linear', C=10, threshold=1e-5):
    objects_count = len(train_data_set) 
    
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    if train_labels.ndim == 1:
        train_labels = train_labels.reshape(-1, 1)

    P = np.dot(train_labels, train_labels.T) * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count).astype(float)
    b = 0.0

    cvxopt.solvers.options['show_progress'] = False 
    sol = cvxopt.solvers.qp(cvxopt.matrix(P), cvxopt.matrix(q), cvxopt.matrix(G), cvxopt.matrix(h), cvxopt.matrix(A), cvxopt.matrix(b))

    lambdas = np.array(sol['x']).flatten()

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels.flatten()[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(lambdas * targets * np.reshape(kernel[support_vectors_id[n], support_vectors_id], (vector_number,)))
    
    # Avoid division by zero if no support vectors are found
    if len(lambdas) > 0:
        b /= len(lambdas)
    else:
        b = 0

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number

def build_kernel(data_set, kernel_type='linear'):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == 'rbf':
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (np.dot((np.diag(kernel)*np.ones((1, objects_count))).T, b.T)
                         + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T))
        kernel = np.exp(kernel / (2. * sigma ** 2))
    return kernel

def classify_rbf(test_data_set, train_data_set, lambdas, targets, b, vector_number, support_vectors, support_vectors_id):
    kernel = np.dot(test_data_set, support_vectors.T)
    sigma = 1.0
    #K = np.dot(test_data_set, support_vectors.T)
    #kernel = build_kernel(train_data_set, kernel_type='rbf')
    c = (1. / sigma * np.sum(test_data_set ** 2, axis=1) * np.ones((1, np.shape(test_data_set)[0]))).T
    c = np.dot(c, np.ones((1, np.shape(kernel)[1])))
    #aa = np.dot((np.diag(K)*np.ones((1,len(test_data_set)))).T[support_vectors_id], np.ones((1, np.shape(K)[0]))).T
    sv = (np.diag(np.dot(train_data_set, train_data_set.T))*np.ones((1,len(train_data_set)))).T[support_vectors_id]
    aa = np.dot(sv,np.ones((1,np.shape(kernel)[0]))).T
    kernel = kernel - 0.5 * c - 0.5 * aa
    kernel = np.exp(kernel / (2. * sigma ** 2))

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    return np.sign(y)

In [35]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split

# We need to load the IRIS dataset
iris = datasets.load_iris()
X = iris.data
y = iris.target

train_data_set, test_data_set, train_labels, test_labels = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_labels = train_labels.astype(float)
test_labels = test_labels.astype(float)

print(f"Training data shape: {train_data_set.shape}")
print(f"Test data shape: {test_data_set.shape}")

Training data shape: (120, 4)
Test data shape: (30, 4)


In [36]:
from sklearn.metrics import accuracy_score

data_set = (train_data_set, test_data_set, train_labels, test_labels)
unique_labels = np.unique(train_labels)
labels_count = get_labels_count(data_set)

decision_values = np.zeros((len(test_data_set), labels_count))

for idx, label in enumerate(unique_labels):
    # get binary datasets for the current label (One-vs-Rest)
    cur_train_data, cur_test_data, cur_train_labels, cur_test_labels = choose_set_for_label(data_set, label)
    
    # train the model for the current label
    lambdas, support_vectors, support_vectors_id, b, targets, vector_number = train(
        cur_train_data, cur_train_labels, kernel_type='rbf'
    )
    
    # calculate raw decision values for the test data
    # (We adapt the logic from classify_rbf but skip np.sign() to resolve ties)
    kernel_test = np.dot(cur_test_data, support_vectors.T)
    sigma = 1.0
    
    c = (1. / sigma * np.sum(cur_test_data ** 2, axis=1) * np.ones((1, np.shape(cur_test_data)[0]))).T
    c = np.dot(c, np.ones((1, np.shape(kernel_test)[1])))
    
    sv = (np.diag(np.dot(cur_train_data, cur_train_data.T)) * np.ones((1, len(cur_train_data)))).T[support_vectors_id]
    aa = np.dot(sv, np.ones((1, np.shape(kernel_test)[0]))).T
    
    kernel_test = kernel_test - 0.5 * c - 0.5 * aa
    kernel_test = np.exp(kernel_test / (2. * sigma ** 2))
    
    y = np.zeros((np.shape(cur_test_data)[0], 1))
    for j in range(np.shape(cur_test_data)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel_test[j, i]
        y[j] += b
        
    decision_values[:, idx] = y.flatten()

predicted_indices = np.argmax(decision_values, axis=1)
predicted = [unique_labels[i] for i in predicted_indices]

print("Accuracy:", accuracy_score(predicted, test_labels))

Accuracy: 1.0
